# Phase 0 — Hello, Earth Engine

**Exit criterion:** authenticate against Earth Engine, load the Mumbai boundary, and render
a Landsat surface-temperature image over the city.

This notebook is a proof of access, not pipeline code. It establishes three things:

1. Earth Engine credentials and project binding work.
2. Landsat Collection 2 Level-2 data is reachable for Mumbai.
3. The surface-temperature values come out in plausible °C — the part most easily got
   wrong, and the part that silently poisons everything downstream if it is.

Phase 1 rewrites all of this as `data-pipeline/` modules. Nothing here is meant to be
reused as-is — notebooks are for exploration (`docs/conventions.md`).

**Terminology.** Outputs are *surface* temperature (LST), never "temperature" unqualified.
This is a standing rule, not a stylistic preference (ADR-0005).

## 0. One-time setup

Run once per machine, from the repo root:

```bash
uv sync                          # creates .venv and installs everything
uv run earthengine authenticate  # opens a browser
```

`earthengine authenticate` signs you in with the Google account that owns the registered
noncommercial project and writes a refresh token to
`%USERPROFILE%\.config\earthengine\credentials`. That file lives outside the repo and is
never committed.

Then select the `.venv` (Python 3.12) kernel in VS Code before running this notebook.

In [3]:
import os
from pathlib import Path

import ee
from dotenv import load_dotenv

# Notebooks read .env directly through python-dotenv. When Phase 1 promotes this work
# into data-pipeline/ modules, config moves to pydantic-settings per docs/conventions.md.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(REPO_ROOT / ".env")

GEE_PROJECT_ID = os.getenv("GEE_PROJECT_ID")
if not GEE_PROJECT_ID:
    raise RuntimeError("GEE_PROJECT_ID is missing from .env — see docs/runbook.md §1.1")

print("Repo root:      ", REPO_ROOT)
print("Earth Engine project:", GEE_PROJECT_ID)

Repo root:       c:\Users\devgu\Downloads\Project\Personal\Repos\major-project-urbanheat
Earth Engine project: urbanheat-mumbai


## 1. Authenticate and initialise

Two separate steps that are easy to confuse:

- **`ee.Authenticate()`** — interactive, once per machine. Runs browser OAuth and caches a
  refresh token on disk.
- **`ee.Initialize(project=...)`** — every session. Exchanges the cached token for an access
  token and binds the session to a Cloud project.

The `project` argument is not optional in practice. The project is what carries the
noncommercial registration and what compute quota is charged against (ADR-0001), which is
why a bare `ee.Initialize()` fails even when authentication succeeded.

The cell below tries `Initialize` first, so a normal run never re-opens a browser.

In [15]:
try:
    ee.Initialize(project=GEE_PROJECT_ID)
except Exception as exc:
    # Broad catch is deliberate: a missing, expired or wrong-account credential surfaces
    # as several different exception types across the ee, google.auth and urllib layers.
    print(f"Initialize failed ({type(exc).__name__}) — starting browser authentication...")
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)

# Smallest possible server round-trip: proves credentials, project binding and network
# in one call, before any real work depends on them.
print("Earth Engine ready —", ee.String("connection ok").getInfo())

Earth Engine ready — connection ok


## 2. The Mumbai boundary

Phase 0 uses **FAO GAUL 2015 level-2** (administrative districts), which already lives in
the Earth Engine catalog — no download, no shapefile wrangling. Greater Mumbai spans *two*
GAUL districts, Mumbai (the island city) and Mumbai Suburban, so both are selected and
dissolved into a single polygon.

This is deliberately a placeholder. Phase 1 replaces it with real BMC ward polygons from
Datameet/OSM (`PROGRESS.md` Phase 1, `data-dictionary.md` §5), because wards are the unit
planners actually work in and GAUL has neither ward boundaries nor BMC's exact extent.

In [16]:
GAUL_L2 = "FAO/GAUL/2015/level2"

# Sanity check before filtering. Names in administrative datasets drift between versions
# ("Mumbai" vs "Greater Bombay"), so print what is actually there rather than trusting a
# guess that would otherwise fail as a silent empty result.
maharashtra = ee.FeatureCollection(GAUL_L2).filter(ee.Filter.eq("ADM1_NAME", "Maharashtra"))

# One getInfo() on a short list of names is fine. The rule in ADR-0001 is never to call
# getInfo() inside a per-cell loop — that is what burns the monthly compute quota.
district_names = maharashtra.aggregate_array("ADM2_NAME").getInfo()

print(f"{len(district_names)} districts in Maharashtra")
print("Candidates:", sorted(n for n in district_names if "umb" in n or "omba" in n))

35 districts in Maharashtra
Candidates: ['Mumbai Suburban', 'Mumbai city']


In [ ]:
# GAUL spells the island city "Mumbai city" — lowercase "c", exactly as printed above.
# Guessing "Mumbai" here matched only the suburban district and silently dropped ~160 km²
# of the densest part of the city. Copy these strings from the output above; never assume
# them, because administrative datasets are inconsistent about capitalisation.
MUMBAI_DISTRICTS = ["Mumbai city", "Mumbai Suburban"]

mumbai_fc = maharashtra.filter(ee.Filter.inList("ADM2_NAME", MUMBAI_DISTRICTS))

# Check the exact count, not merely "not zero". A *partial* match is the dangerous case:
# it yields a valid-looking geometry that is missing part of the study area, and every
# downstream statistic inherits the omission without any error ever being raised.
n_matched = mumbai_fc.size().getInfo()
if n_matched != len(MUMBAI_DISTRICTS):
    matched = mumbai_fc.aggregate_array("ADM2_NAME").getInfo()
    raise RuntimeError(
        f"Expected {len(MUMBAI_DISTRICTS)} districts, matched {n_matched}: {matched}. "
        "Correct MUMBAI_DISTRICTS against the candidates printed above."
    )

# .geometry() dissolves the collection into one geometry: the two districts become a
# single polygon with the internal border removed.
mumbai = mumbai_fc.geometry()

# ee.Geometry.area() returns m² computed on the WGS84 ellipsoid — Earth Engine handles the
# projection internally. The "project to EPSG:32643 for area maths" rule in
# docs/conventions.md applies to local geopandas work in Phase 1, not to server-side EE.
area_km2 = mumbai.area(maxError=1).divide(1e6).getInfo()

# BMC's published area for Greater Mumbai. GAUL is a generalised global product that
# smooths coastlines, so it under-measures a city built largely on reclaimed land. Print
# the gap rather than hide it behind a pass/fail — the size of the gap is the useful part.
BMC_AREA_KM2 = 603

print(f"Matched {n_matched} districts — area {area_km2:,.0f} km²")
print(f"BMC published area:  {BMC_AREA_KM2} km²   ({area_km2 / BMC_AREA_KM2 - 1:+.0%})")
if not 450 < area_km2 < 750:
    print("WARNING: too far off to be a generalisation difference — check the district list")
else:
    print("Right order of magnitude for GAUL. Phase 1's BMC ward polygons close this gap.")

## 3. The Landsat collection

`LANDSAT/LC08/C02/T1_L2` and `LANDSAT/LC09/C02/T1_L2`, decoded:

| Piece | Meaning |
|---|---|
| `LC08` / `LC09` | Landsat 8 / Landsat 9. Same instruments, same band names, orbits 8 days out of phase — using both roughly doubles the observations |
| `C02` | Collection 2, the current reprocessing: improved geolocation, and the surface-temperature product exists at all |
| `T1` | Tier 1, the highest geometric and radiometric quality. Tier 2 is looser and is excluded |
| `L2` | **Level 2** — surface reflectance and surface temperature, atmospherically corrected |

Level 2 is the reason this project is feasible at all. The thermal band arrives as a
retrieved *surface* temperature with emissivity and atmospheric correction already applied
by USGS. Deriving that from Level-1 top-of-atmosphere radiance is a research task in its
own right, and doing it badly is invisible until the model is already built (ADR-0001).

### 3.1 The scale factor — the most common LST mistake

Collection 2 Level-2 bands are stored as **unsigned 16-bit integers**, not floats. Storing
the global 30 m archive as floats would multiply its size, so USGS stores scaled integers
and publishes the constants to undo the scaling.

For the thermal band `ST_B10`:

```
kelvin  = DN × 0.00341802 + 149.0
celsius = kelvin − 273.15
```

Worked through, for a raw pixel value of 44,800:

```
44800 × 0.00341802 + 149.0  =  302.13 K
302.13 − 273.15             =   28.98 °C
```

The failure modes and what each looks like:

| Symptom | Cause |
|---|---|
| Values ≈ 44,000 | No scaling applied — raw DN straight through |
| Values ≈ 300 | Scaled to Kelvin, forgot `− 273.15` |
| Values ≈ 0.15 | Applied the *optical* scale factor to the thermal band |

**The two scale factors are different and are not interchangeable.** Optical bands
`SR_B1`–`SR_B7` use `× 0.0000275 − 0.2` to give reflectance in 0–1; the thermal band uses
the constants above. Mixing them produces numbers plausible enough to survive unnoticed
into a trained model, which is the real danger — an obviously broken number gets caught, a
subtly wrong one gets published.

This is why §4 prints min/mean/max before anything is plotted. `docs/runbook.md` §6 lists
the "≈ 300" symptom for exactly this reason.

### 3.2 Cloud masking with `QA_PIXEL`

Cloud is opaque in the thermal band. Over a cloudy pixel the sensor measures the *cloud
top* — possibly −20 °C — not the ground. One unmasked cloudy observation dragged into a
composite biases that cell cold, and cold errors over Mumbai in April are silent: nothing
about a 24 °C reading looks obviously broken.

Every Collection 2 scene carries a `QA_PIXEL` band — a 16-bit integer per pixel where
**each bit is an independent flag**, set by USGS's own CFMask cloud detection:

| Bit | Flag |
|---|---|
| 0 | Fill (no data) |
| 1 | Dilated cloud |
| 2 | Cirrus |
| 3 | Cloud |
| 4 | Cloud shadow |
| 5 | Snow / ice |
| 6 | Clear |
| 7 | Water |

To test bit *n*, build an integer with only that bit set (`1 << n`) and bitwise-AND it
against the value. A non-zero result means the flag is set:

```
1 << 3   =   0b00001000   =   8      ← the "cloud" bit
qa AND 8 == 0                        ← this pixel is not flagged as cloud
```

**Four bits are rejected here, each for a different reason:**

- **Bit 3 — Cloud.** The obvious one.
- **Bit 4 — Cloud shadow.** Shadowed ground really is cooler at that instant, but it is not
  representative of how that surface behaves thermally. Leaving shadows in biases a cell
  cold in whichever years happened to be shadowed — a bias that moves between cells, which
  is worse than a uniform one.
- **Bit 2 — Cirrus.** Thin high cloud. Frequently invisible in a true-colour preview while
  still attenuating the thermal signal. The dangerous flag, precisely because the scene
  looks clean.
- **Bit 1 — Dilated cloud.** A buffer grown around detected cloud, because cloud edges are
  systematically under-detected. Using it discards some genuinely good pixels to remove
  contaminated ones; with a multi-year median and plenty of observations per cell, that is
  the right side of the trade.

Bit 5 (snow) is irrelevant in Mumbai. Bit 7 (water) is deliberately **not** masked — the
Arabian Sea, Powai and Vihar are legitimately cool surfaces and part of what the model
should see.

**Two layers of filtering doing different jobs**

1. `CLOUD_COVER < 40` — a **scene-level** metadata filter that cheaply discards
   mostly-cloudy scenes before any pixel work happens.
2. The `QA_PIXEL` bitmask — the **per-pixel** decision.

The scene filter is deliberately loose. A Landsat scene is about 185 km across; a scene
reported as 40% cloudy can be completely clear over Mumbai's ~600 km². Filtering hard at
the scene level throws away usable pixels for no gain — the per-pixel mask is what actually
protects the composite.

In [18]:
# --- Collection 2 Level-2 scaling constants (USGS Data Format Control Book) ---
ST_SCALE, ST_OFFSET = 0.00341802, 149.0  # ST_B10 → Kelvin
SR_SCALE, SR_OFFSET = 0.0000275, -0.2  # SR_B1..SR_B7 → reflectance 0–1
KELVIN_TO_C = 273.15

# Listed explicitly rather than by wildcard: SR_QA_AEROSOL must not be scaled as if it
# were reflectance, and an explicit list cannot accidentally catch it.
OPTICAL_BANDS = ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"]

# QA_PIXEL bit positions to reject — see the table above for why each one is here.
CLOUD_BITS = {"dilated_cloud": 1, "cirrus": 2, "cloud": 3, "cloud_shadow": 4}


def prepare(image: ee.Image) -> ee.Image:
    """Scale one Landsat C2 L2 scene to physical units and mask cloud-affected pixels."""
    qa = image.select("QA_PIXEL")

    # Build the keep-mask by ANDing together "this flag is not set" for each bit.
    bits = list(CLOUD_BITS.values())
    clear = qa.bitwiseAnd(1 << bits[0]).eq(0)
    for bit in bits[1:]:
        clear = clear.And(qa.bitwiseAnd(1 << bit).eq(0))

    # Split into two named steps rather than one chain, so both halves of the conversion
    # are visible: the scaled integer becomes Kelvin, and only then becomes Celsius.
    # Skipping the second step is the "values around 300" failure in docs/runbook.md §6.
    kelvin = image.select("ST_B10").multiply(ST_SCALE).add(ST_OFFSET)
    lst_celsius = kelvin.subtract(KELVIN_TO_C).rename("LST")

    optical = image.select(OPTICAL_BANDS).multiply(SR_SCALE).add(SR_OFFSET)

    return ee.Image(
        image.addBands(lst_celsius)
        .addBands(optical, None, True)  # (src, names, overwrite) — replace unscaled originals
        .updateMask(clear)
        # Band arithmetic returns a new image that has dropped the source metadata.
        # system:time_start must be carried forward or date filtering and any per-year
        # work downstream stops working — silently, which is the problem.
        .copyProperties(image, ["system:time_start", "SPACECRAFT_ID", "CLOUD_COVER"])
    )

In [19]:
# Year range and season. The years are still an open question that Phase 1 settles
# (data-dictionary.md §5). The monsoon exclusion is not negotiable — Jun–Sep optical and
# thermal imagery over Mumbai is unusable under cloud (ADR-0005, conventions.md "Gotchas").
START_YEAR, END_YEAR = 2019, 2025
DRY_SEASON = (3, 5)  # March–May inclusive
MAX_SCENE_CLOUD = 40  # percent — scene-level pre-filter only, see §3.2

landsat = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_L2"))
    .filterBounds(mumbai)
    .filterDate(f"{START_YEAR}-01-01", f"{END_YEAR + 1}-01-01")
    # calendarRange keeps Mar–May of *every* year in the range. A plain date range
    # cannot express "these months, repeatedly".
    .filter(ee.Filter.calendarRange(DRY_SEASON[0], DRY_SEASON[1], "month"))
    .filter(ee.Filter.lt("CLOUD_COVER", MAX_SCENE_CLOUD))
)

n_scenes = landsat.size().getInfo()
print(f"{n_scenes} scenes over Mumbai — Mar–May, {START_YEAR}–{END_YEAR}")
if n_scenes == 0:
    raise RuntimeError("No scenes matched. Check the boundary and the date filters above.")

56 scenes over Mumbai — Mar–May, 2019–2025


In [20]:
prepared = landsat.map(prepare)

# Median, not mean. A median resists the handful of cloud pixels the QA mask inevitably
# misses: one missed −20 °C cloud-top pixel visibly drags a multi-year mean, and barely
# moves the median. ADR-0005 specifies median compositing for this reason.
lst = prepared.select("LST").median().clip(mumbai)

print("Composite built (lazily — nothing has been computed on the server yet).")

Composite built (lazily — nothing has been computed on the server yet).


## 4. Check the numbers before looking at the picture

A map renders something regardless of whether the scaling is right — wrong values with a
wrong stretch just look like a different picture. Verify numerically first.

This is a single server-side reduction returning three numbers, which is exactly the
pattern ADR-0001 asks for: aggregate on the server, download the aggregate.

In [21]:
stats = lst.reduceRegion(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
    geometry=mumbai,
    scale=100,  # native thermal resolution; asking for 30 m would only interpolate
    maxPixels=1e9,
    bestEffort=True,
).getInfo()

print(f"min   {stats['LST_min']:6.1f} °C")
print(f"mean  {stats['LST_mean']:6.1f} °C")
print(f"max   {stats['LST_max']:6.1f} °C")

min     29.0 °C
mean    39.8 °C
max     51.6 °C


**What to expect** — Mumbai, dry-season median, ~10:30 local overpass, clipped to the
land boundary. The middle column is what this notebook actually produced over 2019–2025,
not a prediction:

| | Observed | What it is |
|---|---|---|
| min | ~29 °C | The coolest *land* — dense canopy inside Sanjay Gandhi National Park |
| mean | ~40 °C | City-wide land average |
| max | ~52 °C | Bare ground, industrial roofing, the airport apron |

**Why there is no 22–26 °C minimum.** GAUL district polygons are administrative *land*
boundaries, so `.clip(mumbai)` removes the Arabian Sea almost entirely. The minimum here
is therefore the coolest land surface, not water. Clip to a boundary that includes sea and
the minimum drops several degrees — that number is a property of the region you clipped
to, not of the temperature retrieval. Worth knowing before comparing against a published
figure that used a different footprint.

**Sanity thresholds.** If the mean comes out near **313**, `− 273.15` was skipped. Near
**44,000**, no scaling was applied at all. Near **0.15**, the optical scale factor hit the
thermal band. Fix it here — everything downstream inherits this number.

**On that maximum.** A 52 °C surface temperature is not a 52 °C air temperature; Mumbai
air in April sits around 31–33 °C. Asphalt and metal roofing genuinely reach the low
fifties at mid-morning while the air above them does not, and a ~9 °C mean gap between
surface and air is ordinary for a dense tropical city. That gap is the whole substance of
ADR-0005, and it is the single most likely thing for a viva panel to push on — the honest
answer is that this project measures surfaces, and labels every output as such.

## 5. Render

In [22]:
import geemap

# The standard Earth Engine temperature ramp: blue (cool) → green → yellow → red (hot).
# Widely used in the LST literature, so figures here stay comparable to published work.
# Its green band can read as vegetation rather than as temperature, which is why §6 plots
# NDVI as its own layer instead of leaving the two to be conflated in a report figure.
# A colour ramp reads as a ramp when it is laid out as a grid, not as 23 separate lines.
# fmt: off
LST_PALETTE = [
    "040274", "0502a3", "0502ce", "235cb1", "307ef3", "269db1", "30c8e2",
    "32d3ef", "3be285", "86e26f", "b5e22e", "d5ea5f", "fff705", "ffd611",
    "ffb613", "ff8b13", "ff6e08", "ff500d", "ff0000", "de0101", "c21301",
    "a71001", "911003",
]
# fmt: on

# min/max are display stretch only — they change the picture, never the data. Adjust
# them against the statistics printed above.
LST_VIS = {"min": 25, "max": 45, "palette": LST_PALETTE}

m = geemap.Map(center=[19.08, 72.88], zoom=11)
m.add_basemap("CartoDB.DarkMatter")
m.addLayer(lst, LST_VIS, "Surface temperature (°C)")
m.addLayer(ee.Image().paint(mumbai_fc, 0, 2), {"palette": "white"}, "Mumbai boundary")
m.add_colorbar(LST_VIS, label="Land surface temperature (°C)")
m

Map(center=[19.08, 72.88], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

If that cell renders blank or shows a raw widget error, the problem is `ipyleaflet`'s
Jupyter widget layer, not Earth Engine. In order:

1. **Restart the kernel.** Widget extensions do not load into a kernel that was already
   running when they were installed. This fixes it most of the time.
2. **Switch to the folium backend** — replace the import with
   `import geemap.foliumap as geemap`. Folium renders plain HTML with no widget
   dependency. Layer control is less capable, but it always works.

The static PNG below depends on neither, and is the safest thing to put in the report.

### 5.1 Static image — the report figure

`getThumbURL` renders server-side and hands back a PNG. No widgets involved, so this is the
dependable path for a report figure, and it doubles as a check that the interactive map is
showing what you think it is.

In [23]:
from IPython.display import Image as DisplayImage

thumb_url = lst.getThumbURL({**LST_VIS, "region": mumbai, "dimensions": 900, "format": "png"})

print(thumb_url)
DisplayImage(url=thumb_url)

https://earthengine.googleapis.com/v1/projects/urbanheat-mumbai/thumbnails/a103ce04170c5ca0f61f9e67993ccb2c-3bc2b92250ac18c5717e3563e1785f09:getPixels


## 6. Cross-check — does the map agree with physical reality?

A picture that renders is not a picture that is *correct*. The cheapest strong check is
that surface temperature should be visibly **inverse** to vegetation:

- **Cool** — Sanjay Gandhi National Park, Aarey, the Mahalaxmi racecourse, the creeks.
- **Hot** — Dharavi, the eastern industrial belt, the airport, reclaimed bare ground.

If that pattern is absent or reversed, something is wrong with the scaling, the masking or
the boundary — investigate before Phase 1 builds on it.

NDVI here comes from the **Landsat** bands already loaded (`SR_B5` near-infrared, `SR_B4`
red), purely because they are in hand. Phase 1's NDVI comes from **Sentinel-2** at 10 m
(`data-dictionary.md` §3): finer, and independent of the sensor that produced the target,
which matters for the leakage argument in §4 of that file.

In [24]:
# (NIR − Red) / (NIR + Red). Vegetation reflects strongly in NIR and absorbs red, so
# healthy canopy runs high; built surfaces and bare ground sit near zero.
ndvi = prepared.median().normalizedDifference(["SR_B5", "SR_B4"]).rename("NDVI").clip(mumbai)

NDVI_VIS = {
    "min": 0.0,
    "max": 0.6,
    "palette": ["ffffff", "ce7e45", "fcd163", "a3ce59", "207401", "012e01"],
}

m2 = geemap.Map(center=[19.08, 72.88], zoom=11)
m2.addLayer(lst, LST_VIS, "Surface temperature (°C)")
m2.addLayer(ndvi, NDVI_VIS, "NDVI")
m2.add_colorbar(LST_VIS, label="Land surface temperature (°C)")

# Toggle the two layers in the control at the top right — the green areas on NDVI should
# line up with the blue areas on LST.
m2

Map(center=[19.08, 72.88], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

## What this establishes

- Earth Engine credentials, project binding and quota access work end to end.
- Landsat 8/9 Collection 2 Level-2 is reachable and filterable for Mumbai.
- The thermal band decodes to physically plausible °C — verified numerically, not assumed.
- Cloud masking and dry-season compositing ran server-side; nothing large was downloaded,
  which is the whole argument of ADR-0001.

## What Phase 1 does differently

| Here (Phase 0) | Phase 1 |
|---|---|
| GAUL district boundary | Real BMC ward polygons → `data/processed/wards.geojson` |
| A rendered image | A ~200 m grid with a stable `cell_id`, reduced to one row per cell |
| Landsat NDVI, incidentally | Sentinel-2 NDVI/NDBI/NDWI at 10 m, plus WorldCover, WorldPop, SRTM, OSM |
| Notebook code | `data-pipeline/` modules, config via `pydantic-settings` |
| Nothing persisted | `data/processed/features.parquet` |